[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skyexry/urban-mobility-forecast/blob/main/notebooks/10_train_eval.ipynb)

# 10 — Graph Fix (sigma2=0.0005, theta=0.70)

Previous graph (`edge_index.npy`) used `theta=0.90`, compressing all edge weights into [0.90, 1.00].  
ChebConv could not meaningfully differentiate between near and far neighbors.

This notebook rebuilds the graph with `theta=0.70`, giving edge weights in [0.70, 1.00] and saves as `edge_index_v2.npy` / `edge_weight_v2.npy` (original files untouched).

Settings (same as 09 otherwise):
- `INPUT_WINDOW = 168`, `BATCH_SIZE = 16`
- `hidden_channels=32, tcn_channels=64`
- Transformer decoder, `decoder_layers=1`
- `asymmetric_mse_loss(alpha=2.0)`, `patience=20`

## Setup

In [1]:
import os
if os.path.exists('/root/urban-mobility-forecast'):
    BASE_DIR = '/root/urban-mobility-forecast'
    DATA_DIR = '/root/urban-mobility-forecast/data'
else:
    BASE_DIR = '/content/urban-mobility-forecast'
    DATA_DIR = '/content/drive/MyDrive/citibike'
print(f'BASE_DIR: {BASE_DIR}')
print(f'DATA_DIR: {DATA_DIR}')

BASE_DIR: /root/urban-mobility-forecast
DATA_DIR: /root/urban-mobility-forecast/data


In [2]:
!git clone https://github.com/skyexry/urban-mobility-forecast.git 2>/dev/null || git -C urban-mobility-forecast pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 6.32 KiB | 3.16 MiB/s, done.
From https://github.com/skyexry/urban-mobility-forecast
 * [new branch]      graph-fix  -> origin/graph-fix
Already up to date.


In [3]:
import sys
if BASE_DIR == '/content/urban-mobility-forecast':
    from google.colab import drive
    drive.mount('/content/drive')
!pip install torch-geometric -q
sys.path.append(BASE_DIR)

In [4]:
import importlib, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import joblib, warnings
warnings.filterwarnings('ignore')

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

features_mod = load_module('features', f'{BASE_DIR}/preprocessing/features.py')
graph_mod    = load_module('graph',    f'{BASE_DIR}/preprocessing/graph.py')
from model.stgnn import STGNN

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS


Device: cuda


## 1. Load Data

In [5]:
df       = pd.read_parquet(f'{DATA_DIR}/hourly_demand_filtered.parquet')
stations = pd.read_parquet(f'{DATA_DIR}/stations_final.parquet')
df['hour'] = pd.to_datetime(df['hour'])
print(f'Stations   : {df["start_station_id"].nunique()}')
print(f'Date range : {df["hour"].min()} → {df["hour"].max()}')

Stations   : 100
Date range : 2024-01-01 00:00:00 → 2025-12-31 23:00:00


## 2. Rebuild Graph (theta=0.70)

Original: `sigma2=0.0005, theta=0.90` → 656 edges, weights ∈ [0.90, 1.00]  
New:      `sigma2=0.0005, theta=0.70` → ~1900 edges, weights ∈ [0.70, 1.00]

Saved as `edge_index_v2.npy` / `edge_weight_v2.npy` — original files untouched.

In [6]:
SIGMA2 = 0.0005
THETA  = 0.70

W_v2, _ = graph_mod.build_adjacency_matrix(df, sigma2=SIGMA2, theta=THETA)
edge_index_v2, edge_weight_v2 = graph_mod.adjacency_to_edge_index(W_v2)

print(f'edge_index_v2 : {edge_index_v2.shape}')
print(f'edge_weight_v2: {edge_weight_v2.shape}')
print(f'weight range  : min={edge_weight_v2.min():.3f}, mean={edge_weight_v2.mean():.3f}, max={edge_weight_v2.max():.3f}')

np.save(f'{DATA_DIR}/edge_index_v2.npy',  edge_index_v2)
np.save(f'{DATA_DIR}/edge_weight_v2.npy', edge_weight_v2)
print('Saved edge_index_v2.npy and edge_weight_v2.npy')

Stations     : 100
Non-zero edges: 1900
Sparsity     : 81.00%
edge_index_v2 : (2, 1900)
edge_weight_v2: (1900,)
weight range  : min=0.700, mean=0.850, max=0.999
Saved edge_index_v2.npy and edge_weight_v2.npy


In [7]:
edge_index  = torch.tensor(edge_index_v2,  dtype=torch.long).to(DEVICE)
edge_weight = torch.tensor(edge_weight_v2, dtype=torch.float).to(DEVICE)

# Compare with original
ei_old = np.load(f'{DATA_DIR}/edge_index.npy')
ew_old = np.load(f'{DATA_DIR}/edge_weight.npy')
print(f'Original graph : {ei_old.shape[1]} edges, weight [{ew_old.min():.3f}, {ew_old.max():.3f}]')
print(f'New graph (v2) : {edge_index_v2.shape[1]} edges, weight [{edge_weight_v2.min():.3f}, {edge_weight_v2.max():.3f}]')

Original graph : 656 edges, weight [0.900, 0.999]
New graph (v2) : 1900 edges, weight [0.700, 0.999]


## 3. Build Demand Matrix & Time Features

In [8]:
station_ids = stations['start_station_id'].tolist()
demand_matrix, hours = features_mod.build_demand_matrix(df, station_ids)
time_feats = features_mod.build_time_features(pd.Series(hours))
print(f'demand_matrix : {demand_matrix.shape}')
print(f'time_features : {time_feats.shape}')

demand_matrix : (16808, 100)
time_features : (16808, 6)


## 4. Train / Val / Test Split

In [9]:
T      = demand_matrix.shape[0]
split1 = int(T * 0.70)
split2 = int(T * 0.85)

train_demand = demand_matrix[:split1]
val_demand   = demand_matrix[split1:split2]
test_demand  = demand_matrix[split2:]

train_time = time_feats[:split1]
val_time   = time_feats[split1:split2]
test_time  = time_feats[split2:]

normalized_train, scaler = features_mod.normalize_demand(train_demand)
normalized_val  = scaler.transform(np.log1p(val_demand).reshape(-1,1)).reshape(val_demand.shape)
normalized_test = scaler.transform(np.log1p(test_demand).reshape(-1,1)).reshape(test_demand.shape)
joblib.dump(scaler, f'{DATA_DIR}/scaler.pkl')

print(f'Train: {train_demand.shape[0]} steps ({train_demand.shape[0]/24:.0f} days)')
print(f'Val  : {val_demand.shape[0]} steps ({val_demand.shape[0]/24:.0f} days)')
print(f'Test : {test_demand.shape[0]} steps ({test_demand.shape[0]/24:.0f} days)')

Train: 11765 steps (490 days)
Val  : 2521 steps (105 days)
Test : 2522 steps (105 days)


## 5. Sliding Windows & DataLoaders

In [10]:
INPUT_WINDOW  = 168
OUTPUT_WINDOW = 72
BATCH_SIZE    = 16

x_demand_train, x_time_train, y_train = features_mod.build_sliding_windows(normalized_train, train_time, INPUT_WINDOW, OUTPUT_WINDOW)
x_demand_val,   x_time_val,   y_val   = features_mod.build_sliding_windows(normalized_val,   val_time,   INPUT_WINDOW, OUTPUT_WINDOW)
x_demand_test,  x_time_test,  y_test  = features_mod.build_sliding_windows(normalized_test,  test_time,  INPUT_WINDOW, OUTPUT_WINDOW)

class CitiBikeDataset(Dataset):
    def __init__(self, x_demand, x_time, y):
        self.x_demand = torch.tensor(x_demand, dtype=torch.float32)
        self.x_time   = torch.tensor(x_time,   dtype=torch.float32)
        self.y        = torch.tensor(y,         dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x_demand[i], self.x_time[i], self.y[i]

train_loader = DataLoader(CitiBikeDataset(x_demand_train, x_time_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(CitiBikeDataset(x_demand_val,   x_time_val,   y_val),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(CitiBikeDataset(x_demand_test,  x_time_test,  y_test),  batch_size=BATCH_SIZE)
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

Samples  : 11526
x_demand : (11526, 100, 168, 1)
x_time   : (11526, 168, 6)
y        : (11526, 100, 72, 1)
Samples  : 2282
x_demand : (2282, 100, 168, 1)
x_time   : (2282, 168, 6)
y        : (2282, 100, 72, 1)
Samples  : 2283
x_demand : (2283, 100, 168, 1)
x_time   : (2283, 168, 6)
y        : (2283, 100, 72, 1)
Train batches: 721, Val: 143, Test: 143


## 6. Training Utilities

In [11]:
from tqdm import tqdm

def asymmetric_mse_loss(pred, target, alpha=2.0):
    residual = target - pred
    weight   = torch.where(residual > 0,
                           alpha * torch.ones_like(residual),
                           torch.ones_like(residual))
    return (weight * residual ** 2).mean()

def log_cosh_loss(pred, target):
    return torch.log(torch.cosh(pred - target)).mean()

def train_epoch(model, loader, optimizer, loss_fn, forward_fn, epoch=None):
    model.train()
    total = 0
    pbar  = tqdm(loader, desc=f'Epoch {epoch:3d} [train]' if epoch else 'train', leave=False)
    for x_d, x_t, y in pbar:
        x_d, x_t, y = x_d.to(DEVICE), x_t.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = forward_fn(model, x_d, x_t)
        loss = loss_fn(pred, y.squeeze(-1))
        loss.backward()
        optimizer.step()
        total += loss.item()
        pbar.set_postfix({'batch_loss': f'{loss.item():.4f}'})
    return total / len(loader)

def eval_epoch(model, loader, forward_fn):
    model.eval()
    mae_total, mse_total = 0, 0
    with torch.no_grad():
        for x_d, x_t, y in tqdm(loader, desc='         [val]  ', leave=False):
            x_d, x_t, y = x_d.to(DEVICE), x_t.to(DEVICE), y.to(DEVICE)
            pred = forward_fn(model, x_d, x_t)
            diff = pred - y.squeeze(-1)
            mae_total += diff.abs().mean().item()
            mse_total += (diff ** 2).mean().item()
    n = len(loader)
    return mae_total / n, (mse_total / n) ** 0.5

def run_training(model, train_loader, val_loader, forward_fn,
                 lr=1e-3, epochs=100, patience=20,
                 save_path=None, checkpoint_every=None, loss_fn=None):
    if loss_fn is None:
        loss_fn = log_cosh_loss
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)
    best_val, wait, history = float('inf'), 0, []

    for epoch in range(1, epochs + 1):
        tr_loss         = train_epoch(model, train_loader, optimizer, loss_fn, forward_fn, epoch=epoch)
        vl_mae, vl_rmse = eval_epoch(model, val_loader, forward_fn)
        history.append((tr_loss, vl_mae, vl_rmse))
        scheduler.step(vl_mae)
        lr_now = optimizer.param_groups[0]['lr']

        print(f'Epoch {epoch:3d} | loss {tr_loss:.4f} | val MAE {vl_mae:.4f} | val RMSE {vl_rmse:.4f} | best {best_val:.4f} | patience {wait}/{patience} | lr {lr_now:.2e}')

        if vl_mae < best_val:
            best_val, wait = vl_mae, 0
            if save_path:
                torch.save(model.state_dict(), save_path)
        else:
            wait += 1
            if wait >= patience:
                print(f'→ Early stop @ epoch {epoch}')
                break

        if checkpoint_every and epoch % checkpoint_every == 0 and save_path:
            ckpt = save_path.replace('.pth', f'_ckpt{epoch}.pth')
            torch.save(model.state_dict(), ckpt)

    return history

def compute_metrics(y_true, y_pred, scaler):
    y_true = features_mod.inverse_transform_demand(y_true, scaler)
    y_pred = features_mod.inverse_transform_demand(y_pred, scaler)
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return dict(MAE=mae, RMSE=rmse)

def get_predictions(model, loader, forward_fn):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for x_d, x_t, y in tqdm(loader, desc='Predicting', leave=False):
            x_d, x_t = x_d.to(DEVICE), x_t.to(DEVICE)
            preds.append(forward_fn(model, x_d, x_t).cpu().numpy())
            trues.append(y.squeeze(-1).numpy())
    return np.concatenate(preds), np.concatenate(trues)

## 7. Train STGNN with v2 Graph

In [12]:
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

stgnn_model   = STGNN(num_nodes=100, input_window=INPUT_WINDOW, output_window=OUTPUT_WINDOW,
                      hidden_channels=32, tcn_channels=64,
                      use_transformer_decoder=True, decoder_layers=1).to(DEVICE)
stgnn_forward = lambda model, x_d, x_t: model(x_d, x_t, edge_index, edge_weight)
print(f'STGNN parameters: {sum(p.numel() for p in stgnn_model.parameters()):,}')

history_stgnn = run_training(
    stgnn_model, train_loader, val_loader, stgnn_forward,
    loss_fn=asymmetric_mse_loss,
    patience=20,
    save_path=f'{DATA_DIR}/stgnn_10_best.pth',
    checkpoint_every=10
)

STGNN parameters: 470,529


Epoch   1 | loss 0.1759 | val MAE 0.2001 | val RMSE 0.2693 | best inf | patience 0/20 | lr 1.00e-03


Epoch   2 | loss 0.0779 | val MAE 0.1872 | val RMSE 0.2562 | best 0.2001 | patience 0/20 | lr 1.00e-03


Epoch   3 | loss 0.0673 | val MAE 0.1726 | val RMSE 0.2390 | best 0.1872 | patience 0/20 | lr 1.00e-03


Epoch   4 | loss 0.0637 | val MAE 0.1842 | val RMSE 0.2520 | best 0.1726 | patience 0/20 | lr 1.00e-03


...

Epoch   7 | loss 0.0599 | val MAE 0.1855 | val RMSE 0.2525 | best 0.1695 | patience 0/20 | lr 1.00e-03


Epoch   8 | loss 0.0586 | val MAE 0.1744 | val RMSE 0.2427 | best 0.1695 | patience 1/20 | lr 1.00e-03


Epoch   9 | loss 0.0577 | val MAE 0.1721 | val RMSE 0.2388 | best 0.1695 | patience 2/20 | lr 1.00e-03


Epoch  10 | loss 0.0566 | val MAE 0.1674 | val RMSE 0.2331 | best 0.1695 | patience 3/20 | lr 1.00e-03


KeyboardInterrupt: 

In [13]:
stgnn_model   = STGNN(num_nodes=100, input_window=INPUT_WINDOW, output_window=OUTPUT_WINDOW,
                      hidden_channels=32, tcn_channels=64,
                      use_transformer_decoder=True, decoder_layers=1).to(DEVICE)
stgnn_forward = lambda model, x_d, x_t: model(x_d, x_t, edge_index, edge_weight)
stgnn_model.load_state_dict(torch.load(f'{DATA_DIR}/stgnn_10_best.pth', map_location=DEVICE))
preds_stgnn, trues_stgnn = get_predictions(stgnn_model, test_loader, stgnn_forward)
metrics_stgnn = compute_metrics(trues_stgnn, preds_stgnn, scaler)
print('STGNN (10):', {k: f"{v:.4f}" for k, v in metrics_stgnn.items()})

STGNN (10): {'MAE': '8.4380', 'RMSE': '13.5423'}


## 8. Training Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
tr = [h[0] for h in history_stgnn]
vl = [h[1] for h in history_stgnn]
ep = range(1, len(history_stgnn) + 1)
ax2 = ax.twinx()
l1, = ax.plot(ep, pd.Series(vl).ewm(span=5).mean(),
              color='darkorange', linewidth=1.8, label='Val MAE')
l2, = ax2.plot(ep, tr, color='steelblue', linestyle='--', linewidth=1.5, label='Train Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Val MAE', color='darkorange')
ax2.set_ylabel('Train Asymmetric MSE', color='steelblue')
ax.tick_params(axis='y', labelcolor='darkorange')
ax2.tick_params(axis='y', labelcolor='steelblue')
ax.legend([l1, l2], ['Val MAE', 'Train Loss'], loc='upper right', fontsize=9)
plt.title('Training Curve — 10 (v2 graph, theta=0.70, Asymmetric MSE)')
plt.tight_layout()
plt.show()

## 9. Results — Compare with 09

In [ ]:
rows = [{'Model': 'STGNN (10 — v2 graph)', **metrics_stgnn}]

pth_09 = f'{DATA_DIR}/stgnn_09_best.pth'
if os.path.exists(pth_09):
    # Load 09 with original graph for fair metric comparison
    ei_old = torch.tensor(np.load(f'{DATA_DIR}/edge_index.npy'),  dtype=torch.long).to(DEVICE)
    ew_old = torch.tensor(np.load(f'{DATA_DIR}/edge_weight.npy'), dtype=torch.float).to(DEVICE)
    stgnn_09 = STGNN(num_nodes=100, input_window=INPUT_WINDOW, output_window=OUTPUT_WINDOW,
                     hidden_channels=32, tcn_channels=64,
                     use_transformer_decoder=True, decoder_layers=1).to(DEVICE)
    stgnn_09.load_state_dict(torch.load(pth_09, map_location=DEVICE))
    fwd_09 = lambda model, x_d, x_t: model(x_d, x_t, ei_old, ew_old)
    preds_09, trues_09 = get_predictions(stgnn_09, test_loader, fwd_09)
    metrics_09 = compute_metrics(trues_09, preds_09, scaler)
    rows.insert(0, {'Model': 'STGNN (09 — orig graph)', **metrics_09})

results = pd.DataFrame(rows).set_index('Model').round(4)
print(results.to_string())
results

## 10. Forecast Visualization

In [ ]:
test_mean  = test_demand.mean(axis=0)
high_idx   = int(test_mean.argsort()[-1])
mid_idx    = int(test_mean.argsort()[50])
low_idx    = int(test_mean.argsort()[5])
sample_idx = 0

preds_r = features_mod.inverse_transform_demand(preds_stgnn, scaler)
trues_r = features_mod.inverse_transform_demand(trues_stgnn, scaler)

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for ax, s_idx, label in zip(axes,
    [high_idx, mid_idx, low_idx],
    [f'High-demand  (station {station_ids[high_idx]})',
     f'Mid-demand   (station {station_ids[mid_idx]})',
     f'Low-demand   (station {station_ids[low_idx]})']):
    t = trues_r[sample_idx, s_idx]
    p = preds_r[sample_idx, s_idx]
    ax.plot(t, color='steelblue',  linewidth=2,   label='Actual')
    ax.plot(p, color='darkorange', linewidth=1.8, linestyle='--', label='STGNN 10 (v2 graph)')
    if os.path.exists(pth_09):
        p09 = features_mod.inverse_transform_demand(preds_09, scaler)[sample_idx, s_idx]
        ax.plot(p09, color='forestgreen', linewidth=1.5, linestyle=':', label='STGNN 09 (orig graph)')
    ax.fill_between(range(OUTPUT_WINDOW), t, p, alpha=0.08, color='grey')
    ax.set_ylabel('Trips/hr'); ax.set_title(label)
    ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xticks(range(0, OUTPUT_WINDOW + 1, 6))
axes[-1].set_xticklabels([f'+{h}h' for h in range(0, OUTPUT_WINDOW + 1, 6)])
axes[-1].set_xlabel('Hours into forecast')
plt.suptitle('STGNN (10: v2 graph theta=0.70) — 72-Hour Demand Forecast vs Actual', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Error Analysis

In [ ]:
from matplotlib.patches import Patch

hours_hod  = hours.hour.values
n_samples  = preds_r.shape[0]
i_idx      = np.arange(n_samples)[:, None]
t_idx      = np.arange(OUTPUT_WINDOW)[None, :]
step_hod   = hours_hod[split2 + INPUT_WINDOW + i_idx + t_idx]
err_step   = np.abs(preds_r - trues_r).mean(axis=1)
hourly_mae = np.array([err_step[step_hod == h].mean() for h in range(24)])

peak   = [7, 8, 9, 17, 18, 19]
colors = ['#e74c3c' if h in peak else '#3498db' for h in range(24)]

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

ax = axes[0]
ax.bar(range(24), hourly_mae, color=colors, edgecolor='white')
ax.set_xlabel('Hour of Day'); ax.set_ylabel('MAE (trips/hr)')
ax.set_title('Forecast Error by Hour of Day'); ax.set_xticks(range(24))
peak_mae    = np.mean([hourly_mae[h] for h in peak])
offpeak_mae = np.mean([hourly_mae[h] for h in range(24) if h not in peak])
ax.text(0.98, 0.95, f'Peak: {peak_mae:.2f}\nOff-peak: {offpeak_mae:.2f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
ax.legend(handles=[Patch(color='#e74c3c', label='Peak (7–9am, 5–7pm)'),
                   Patch(color='#3498db', label='Off-peak')], loc='upper left')

ax2 = axes[1]
station_mae_arr = np.abs(preds_r - trues_r).mean(axis=(0, 2))
sorted_idx = station_mae_arr.argsort()
ax2.bar(range(100), station_mae_arr[sorted_idx], color='steelblue', alpha=0.8)
ax2.axhline(station_mae_arr.mean(), color='darkorange', linestyle='--', linewidth=1.5,
            label=f'Mean = {station_mae_arr.mean():.2f}')
ax2.set_xlabel('Station (sorted by MAE)'); ax2.set_ylabel('MAE (trips/hr)')
ax2.set_title('Per-Station MAE (Test Set)'); ax2.legend()

plt.suptitle('STGNN (10) — Error Analysis', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Peak hour MAE:    {peak_mae:.2f} trips/hr')
print(f'Off-peak MAE:     {offpeak_mae:.2f} trips/hr')